# 17 - Fault injection and detection latency

Faults injected into the Docker test environment while the detection service runs in live mode, and the processing time per model in replay mode.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.fault_latency import detection_results, false_alerts, load_alerts, load_runs, scenario_summary
from src.streaming.benchmark import processing_times

DOCS, PROCESSED, MODELS = ROOT / "docs", ROOT / "data" / "processed", ROOT / "models"
RUNS, ALERTS = DOCS / "fault_runs.csv", PROCESSED / "fault_alerts.jsonl"
pd.set_option("display.max_colwidth", 120)

## Live packs

Rebuilt on the 90-minute normal run (`python -m src.streaming.packs live`): 20 Locust users and 60 successful SSH logins, collected with `scripts/collect_logs.py`. 1-minute windows; PCA and Isolation Forest at the 99th percentile of the training scores, LSTM at k = 9.

In [2]:
pd.DataFrame([
    {"service": s, "training windows": p["train_windows"], "templates": len(p["event_cols"]), "models": ", ".join(p["detectors"])}
    for s in ["nginx", "app", "postgres", "ssh"] for p in [joblib.load(MODELS / f"stream_{s}.joblib")]
]).style.hide(axis="index")

service,training windows,templates,models
nginx,91,3,"PCA, Isolation Forest, LSTM"
app,91,3,"PCA, Isolation Forest, LSTM"
postgres,91,9,"PCA, Isolation Forest, LSTM"
ssh,48,2,"PCA, Isolation Forest, LSTM"


## Runs

`scripts/inject_faults.py --runs 5`: database failure (postgres stopped for 60 s), traffic overload (300 extra Locust users for 60 s) and password guessing (30 failed SSH logins in 60 s), in rotation, with 7 minutes between the end of one fault and the start of the next. Normal traffic (20 users, occasional SSH logins) runs the whole time.

In [3]:
faults, experiment_start, experiment_end = load_runs(RUNS)
print(f"Experiment: {experiment_start:%Y-%m-%d %H:%M:%S} - {experiment_end:%H:%M:%S} UTC, {len(faults)} runs")
faults[["run", "scenario", "affected", "fault_start", "fault_end", "details"]].style.hide(axis="index")

Experiment: 2026-11-22 11:51:04 - 13:55:59 UTC, 15 runs


run,scenario,affected,fault_start,fault_end,details
1,db,"postgres,app,nginx",2026-11-22 11:56:04.844000+00:00,2026-11-22 11:57:05.101000+00:00,"postgres stopped, started again after 60 s"
2,overload,"nginx,app,postgres",2026-11-22 12:04:05.101000+00:00,2026-11-22 12:05:05.405000+00:00,300 extra Locust users for 60 s (normal: 20)
3,ssh,ssh,2026-11-22 12:12:05.406000+00:00,2026-11-22 12:13:03.532000+00:00,"30 attempts: 14 failed passwords, 16 connections refused"
4,db,"postgres,app,nginx",2026-11-22 12:20:03.532000+00:00,2026-11-22 12:21:03.884000+00:00,"postgres stopped, started again after 60 s"
5,overload,"nginx,app,postgres",2026-11-22 12:28:03.886000+00:00,2026-11-22 12:29:04.545000+00:00,300 extra Locust users for 60 s (normal: 20)
6,ssh,ssh,2026-11-22 12:36:04.546000+00:00,2026-11-22 12:37:02.679000+00:00,"30 attempts: 14 failed passwords, 16 connections refused"
7,db,"postgres,app,nginx",2026-11-22 12:44:02.680000+00:00,2026-11-22 12:45:02.998000+00:00,"postgres stopped, started again after 60 s"
8,overload,"nginx,app,postgres",2026-11-22 12:52:02.998000+00:00,2026-11-22 12:53:03.564000+00:00,300 extra Locust users for 60 s (normal: 20)
9,ssh,ssh,2026-11-22 13:00:03.564000+00:00,2026-11-22 13:01:01.695000+00:00,"30 attempts: 14 failed passwords, 16 connections refused"
10,db,"postgres,app,nginx",2026-11-22 13:08:01.695000+00:00,2026-11-22 13:09:01.999000+00:00,"postgres stopped, started again after 60 s"


## Detection and latency

Detected: an **Alert** for one of the affected services within 3 minutes of the fault start. Latency: first such alert - fault start. Live windows are 1 minute long and are scored when the minute ends, so the latency includes waiting for the window to close. The `_any` columns also count 'New template' warnings.

In [4]:
alerts = load_alerts(ALERTS)
results = detection_results(faults, alerts, within=180)
results.round(1).to_csv(DOCS / "latency_results.csv", index=False)
results.drop(columns=["fault_end"]).style.format({"latency_s": "{:.0f}", "latency_any_s": "{:.0f}"}, na_rep="-").hide(axis="index")

/tmp/ipykernel_203771/162139729.py:3: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  results.round(1).to_csv(DOCS / "latency_results.csv", index=False)


run,scenario,fault_start,detected,latency_s,first_alert_service,models_fired,alerts,new_template_warnings,detected_any,latency_any_s,levels
1,db,2026-11-22 11:56:04.844000+00:00,True,55,app,"Isolation Forest, PCA",4,2,True,55,"Alert, New template"
2,overload,2026-11-22 12:04:05.101000+00:00,True,55,app,PCA,5,0,True,55,Alert
3,ssh,2026-11-22 12:12:05.406000+00:00,True,56,ssh,"Isolation Forest, PCA",2,0,True,56,Alert
4,db,2026-11-22 12:20:03.532000+00:00,True,58,nginx,"Isolation Forest, PCA",4,2,True,58,"Alert, New template"
5,overload,2026-11-22 12:28:03.886000+00:00,True,56,nginx,PCA,5,0,True,56,Alert
6,ssh,2026-11-22 12:36:04.546000+00:00,True,56,ssh,Isolation Forest,2,0,True,56,Alert
7,db,2026-11-22 12:44:02.680000+00:00,True,60,postgres,"Isolation Forest, PCA",4,2,True,60,"Alert, New template"
8,overload,2026-11-22 12:52:02.998000+00:00,True,57,nginx,PCA,5,0,True,57,Alert
9,ssh,2026-11-22 13:00:03.564000+00:00,True,58,ssh,Isolation Forest,2,0,True,58,Alert
10,db,2026-11-22 13:08:01.695000+00:00,True,60,postgres,"Isolation Forest, PCA",4,2,True,60,"Alert, New template"


In [5]:
summary = scenario_summary(results)
summary.style.format({"detection_rate": "{:.0%}"} | {c: "{:.0f}" for c in summary.columns if c.endswith("_s")}, na_rep="-").hide(axis="index")

scenario,runs,detected,detection_rate,median_latency_s,min_latency_s,max_latency_s,detected_incl_new_template,median_latency_incl_new_template_s
db,5,5,100%,60,55,60,5,60
overload,5,5,100%,57,55,59,5,57
ssh,5,5,100%,58,56,64,5,58


Which models fired, per scenario (number of runs in which each model raised an Alert):

In [6]:
fired = results.assign(model=results["models_fired"].str.split(", ")).explode("model")
fired = fired[fired["model"].fillna("") != ""]
pd.crosstab(fired["scenario"], fired["model"]).reindex(results["scenario"].unique(), fill_value=0)

model,Isolation Forest,PCA
scenario,,
db,5,5
overload,0,5
ssh,5,1


## False alerts during normal traffic

Normal periods: 1-minute windows that overlap no fault (fault start to 2 minutes after its end). The first two minutes of the experiment are skipped.

In [7]:
normal_alerts, normal_hours, fa = false_alerts(faults, alerts, experiment_start, experiment_end, recovery=120)
total = fa.groupby("level")["alerts"].sum()
print(f"Normal traffic: {normal_hours:.2f} h")
for level in ["Alert", "New template"]:
    n = int(total.get(level, 0))
    print(f"{level:13s} {n:4d} in normal periods = {n / normal_hours:.1f} per hour")
fa.style.format({"per_hour": "{:.1f}"}).hide(axis="index")

Normal traffic: 1.03 h
Alert            1 in normal periods = 1.0 per hour
New template     0 in normal periods = 0.0 per hour


source,level,alerts,per_hour
ssh,Alert,1,1.0


Models behind the false Alerts:

In [8]:
fa_models = normal_alerts[normal_alerts["level"] == "Alert"].explode("flagged_by")
pd.crosstab(fa_models["source"], fa_models["flagged_by"])

flagged_by,Isolation Forest,PCA
source,,
ssh,1,1


## Processing time per model (replay mode)

2,000 random test windows per dataset. Batch 1 is how live windows are scored (one at a time); batch 500 is how the service scores many windows that close together (e.g. at the end of an HDFS replay). Drain parsing is per line, explanations per alert.

In [9]:
times = processing_times(n_windows=2000, n_lines=20000)
times.round(4).to_csv(DOCS / "processing_time.csv", index=False)
per_window = times[times["unit"] == "per window"].pivot_table(index=["dataset", "batch_size"], columns="step", values="ms", sort=False)
per_window = per_window[["encode", "PCA", "Isolation Forest", "Random Forest", "LSTM"]]
per_window.assign(total=per_window.sum(axis=1)).round(3)

step                encode    PCA  Isolation Forest  Random Forest   LSTM  \
dataset batch_size                                                          
HDFS    1.0          0.021  0.285             7.515          7.927  0.823   
        500.0        0.006  0.004             0.022          0.021  0.660   
BGL     1.0          0.027  0.317             7.628         10.023  0.912   
        500.0        0.008  0.031             0.038          0.061  0.841   

step                 total  
dataset batch_size          
HDFS    1.0         16.570  
        500.0        0.713  
BGL     1.0         18.907  
        500.0        0.979

In [10]:
times[times["unit"] != "per window"][["dataset", "step", "unit", "ms", "n"]].round(3).style.hide(axis="index")

dataset,step,unit,ms,n
HDFS,Drain parsing,per line,0.024000,20000
HDFS,explanation (Isolation Forest alert),per alert,0.595000,20
HDFS,explanation (Random Forest alert),per alert,27.526000,20
HDFS,explanation (LSTM alert),per alert,0.557000,7
HDFS,explanation (PCA alert),per alert,0.534000,2
BGL,Drain parsing,per line,0.020000,20000
BGL,explanation (LSTM alert),per alert,3.787000,20
BGL,explanation (PCA alert),per alert,3.534000,20
BGL,explanation (Isolation Forest alert),per alert,3.474000,20
BGL,explanation (Random Forest alert),per alert,65.841000,6
